# TP3 : un CNN pour classer les images CIFAR-10

Objectif : reprendre la tâche du TP2 avec un réseau convolutif. Vous construisez le pipeline complet (transforms, ImageFolder, DataLoader), un CNN de trois blocs, vous visualisez ce que détecte sa première couche, puis vous mesurez le gain par rapport au MLP du TP2, sur les mêmes images.

L'énoncé complet, avec les critères de réussite, est dans `enonce.md`.

*Complétez les cellules marquées `# TODO`, dans l'ordre.*

## Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

## Préparation (fournie)

**Données** : CIFAR-10 (Krizhevsky, Nair et Hinton, 2009), fourni par torchvision : des photos couleur de 32 x 32 pixels, en 10 classes (avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion). Le site du jeu ne donne pas de licence ; il demande de citer le rapport *Learning Multiple Layers of Features from Tiny Images* (Krizhevsky, 2009). Tous les TP utilisent les mêmes images : 5000 d'entraînement et 1000 de validation, tirées de l'entraînement de CIFAR-10, et 1000 de test.

Les images sont rangées au format ImageFolder : `cifar10/<partie>/<classe>/`.

In [ ]:
def preparer_archive_cifar10(racine="data"):
    """Place les fichiers de CIFAR-10 dans racine/cifar-10-batches-py sans passer par le serveur d'origine, lent.

    Dans l'ordre : fichiers déjà décompressés ; archive déposée dans la session (cifar-10-python.tar.gz ou
    cifar-10-batches-py.tar.gz, dans le dossier du notebook ou dans racine) ; archive d'origine copiée sur le
    Google Drive de la formation. Si rien n'aboutit, torchvision télécharge ensuite depuis le serveur d'origine ;
    dans tous les cas, il vérifie l'intégrité des fichiers.
    """
    import tarfile
    import urllib.request
    from pathlib import Path

    if Path(racine, "cifar-10-batches-py").exists():
        return
    noms = ("cifar-10-python.tar.gz", "cifar-10-batches-py.tar.gz")
    deposees = [Path(dossier, nom) for dossier in (".", racine) for nom in noms if Path(dossier, nom).exists()]
    if deposees:
        archive = deposees[0]
        print(f"Archive trouvée : {archive}")
    else:
        archive = Path(racine, "cifar-10-python.tar.gz")
        archive.parent.mkdir(parents=True, exist_ok=True)
        url = ("https://drive.usercontent.google.com/download?id=1kyq-1VhOPNr2Bo0w5DiLYuNghOdOWl0a"
               "&export=download&confirm=t")
        print("Téléchargement de CIFAR-10 depuis le Google Drive de la formation (170 Mo)...")
        try:
            urllib.request.urlretrieve(url, archive)
        except OSError as erreur:
            print(f"Google Drive indisponible ({erreur}) : téléchargement depuis le serveur d'origine.")
            return
    try:
        with tarfile.open(archive) as contenu:
            contenu.extractall(racine, filter="data")
    except tarfile.TarError:
        print(f"{archive} n'est pas une archive valide : téléchargement depuis le serveur d'origine.")


def charger_cifar10(dossier="cifar10", n_train=5000, n_val=1000, n_test=1000, seed=0):
    """Images CIFAR-10 au format ImageFolder : dossier/<partie>/<classe>/<image>.png, classes en français.

    Tirages à graine fixe, les mêmes pour tous les TP : n_train et n_val images prises dans les 50 000 images
    d'entraînement de CIFAR-10, sans recouvrement, et n_test dans les 10 000 images de test. Une partie déjà
    présente n'est pas réécrite ; l'écriture passe par un dossier temporaire propre à chaque exécution, renommé à
    la fin : une écriture interrompue n'est jamais prise pour une partie complète, et deux notebooks lancés en même
    temps sur la même machine ne se gênent pas. Les fichiers de CIFAR-10 sont d'abord
    récupérés par preparer_archive_cifar10 (archive déposée dans la session, ou Google Drive de la formation).
    """
    import shutil
    import tempfile
    from pathlib import Path

    import torch
    from torchvision.datasets import CIFAR10

    noms = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
    generateur = torch.Generator().manual_seed(seed)
    ordre = torch.randperm(50000, generator=generateur).tolist()
    parties = {"train": (True, ordre[:n_train]),
               "val": (True, ordre[n_train:n_train + n_val]),
               "test": (False, torch.randperm(10000, generator=generateur)[:n_test].tolist())}
    preparer_archive_cifar10()
    for partie, (entrainement, indices) in parties.items():
        if Path(dossier, partie).exists():
            continue
        source = CIFAR10("data", train=entrainement, download=True)
        Path(dossier).mkdir(parents=True, exist_ok=True)
        temporaire = Path(tempfile.mkdtemp(prefix=f"{partie}.tmp", dir=dossier))
        for numero, indice in enumerate(indices):
            image, label = source[indice]
            sortie = Path(temporaire, noms[label], f"{numero:05d}.png")
            sortie.parent.mkdir(parents=True, exist_ok=True)
            image.save(sortie)
        try:
            temporaire.rename(Path(dossier, partie))
        except OSError:
            # Un autre notebook a fini d'écrire cette partie pendant ce temps : on garde la sienne.
            if not Path(dossier, partie).exists():
                raise
            shutil.rmtree(temporaire)

In [ ]:
charger_cifar10()

## Étape 0 : découvrir les données (fournie)

In [ ]:
from pathlib import Path

for partie in ("train", "val", "test"):
    effectifs = {c.name: len(list(c.iterdir())) for c in sorted(Path(f"cifar10/{partie}").iterdir())}
    print(partie, sum(effectifs.values()), effectifs)

Question : les classes sont-elles équilibrées ? Quelle accuracy obtiendrait un modèle qui répond toujours la même classe ?

*Votre réponse :*

## Étape 1 : transforms, ImageFolder et DataLoader

`conversion` et `eval_tf` sont fournis : tenseur flottant, normalisation des trois canaux. Écrivez `train_tf` : avec `v2.RandomHorizontalFlip`, un miroir horizontal une fois sur deux, puis avec `v2.RandomCrop(32, padding=4)`, un décalage de 4 pixels au plus, puis la conversion. Ici le miroir est réaliste : un chat retourné reste un chat.

In [ ]:
import torch
from torchvision.transforms import v2

conversion = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=[0.5] * 3, std=[0.5] * 3)]
eval_tf = v2.Compose(conversion)

In [ ]:
# TODO : v2.RandomHorizontalFlip(), puis v2.RandomCrop(32, padding=4), puis conversion
train_tf = ...

Créez les trois `ImageFolder` : `train_tf` pour l'entraînement, `eval_tf` pour la validation et le test.

In [ ]:
from torchvision.datasets import ImageFolder

# TODO : trois ImageFolder (cifar10/train, cifar10/val, cifar10/test)
train_ds = ...
val_ds = ...
test_ds = ...
print(train_ds.class_to_idx)
assert len(train_ds.classes) == 10 and train_ds.class_to_idx["automobile"] == 0

Créez les trois DataLoader, de batch 32, avec mélange pour l'entraînement seulement.

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(0)
# TODO : trois DataLoader, batch_size=32, shuffle=True seulement pour l'entraînement
train_loader = ...
val_loader = ...
test_loader = ...
images, labels = next(iter(train_loader))
assert images.shape == (32, 3, 32, 32)

## Étape 2 : le CNN

Trois blocs `Conv2d` (kernel 3, padding 1) + `ReLU` + `MaxPool2d(2)`, avec 16, 32 puis 64 canaux, puis un classifieur : `Flatten`, `Linear(?, 64)`, `ReLU`, `Linear(64, 10)`. Que devient une image 32 x 32 après trois poolings ?

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CNNCifar(nn.Module):
    def __init__(self):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        # TODO : bloc2 et bloc3, sur le modèle de bloc1, avec 32 puis 64 canaux
        self.bloc2 = ...
        self.bloc3 = ...
        # TODO : Flatten, Linear(64 * ? * ?, 64), ReLU, Linear(64, 10)
        self.classifieur = ...

    def forward(self, x):
        return self.classifieur(self.bloc3(self.bloc2(self.bloc1(x))))

In [ ]:
torch.manual_seed(0)
model = CNNCifar().to(device)
assert model(images.to(device)).shape == (32, 10)
print(sum(p.numel() for p in model.parameters()), "paramètres")

## Étape 3 : entraîner

Les fonctions d'entraînement sont fournies.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    for x, y in loader:
        loss = loss_fn(model(x.to(device)), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Entraînez 10 époques avec Adam (lr=1e-3), en affichant à chaque époque la loss et l'accuracy d'entraînement et de validation. Les lignes `writer` (fournies) écrivent les mêmes mesures pour TensorBoard.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

loss_fn = nn.CrossEntropyLoss()
# TODO : optimiseur Adam, lr=1e-3
optimizer = ...
writer = SummaryWriter("runs/tp3")
for epoch in range(1, 11):
    # TODO : une époque d'entraînement, puis evaluate sur train_loader (train_loss, train_acc)
    #        et sur val_loader (val_loss, val_acc)
    writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
    print(f"époque {epoch}  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
writer.close()
assert val_acc > 0.35, "le CNN devrait dépasser 35 % en validation"

Les courbes de l'entraînement dans TensorBoard (fourni), comme dans la démo 5 : les lignes `writer` les ont écrites pendant la boucle.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/tp3

## Étape 4 : ce que voit la première couche

Pour une image d'avion et une image de chat, affichez les 8 premières feature maps de `bloc1` (la convolution suivie de la ReLU).

In [ ]:
import matplotlib.pyplot as plt

model.eval()
fig, axes = plt.subplots(2, 9, figsize=(18, 4))
for ligne, classe in enumerate(("avion", "chat")):
    image, _ = val_ds[val_ds.targets.index(val_ds.class_to_idx[classe])]
    # TODO : sous torch.no_grad(), appliquer model.bloc1[0] puis model.bloc1[1] à image.unsqueeze(0) sur device,
    #        et ramener le résultat sur le CPU dans maps
    maps = ...
    axes[ligne, 0].imshow(image.permute(1, 2, 0) * 0.5 + 0.5)
    axes[ligne, 0].set_title(classe)
    for k in range(8):
        axes[ligne, k + 1].imshow(maps[0, k], cmap="gray")
    for ax in axes[ligne]:
        ax.axis("off")
plt.show()

## Étape 5 : le CNN face au MLP du TP2 (fourni)

On réentraîne ici un MLP identique à la baseline du TP2 (images aplaties en 3072 valeurs), avec la boucle de ce TP : son accuracy peut différer un peu de celle obtenue au TP2. Puis on compare les deux modèles sur la même validation.

In [ ]:
from torch.utils.data import TensorDataset

vers_vecteur = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])


def charger_tenseurs(partie):
    dataset = ImageFolder(f"cifar10/{partie}", transform=vers_vecteur)
    return TensorDataset(torch.stack([image for image, _ in dataset]).flatten(1), torch.tensor(dataset.targets))

In [ ]:
torch.manual_seed(0)
mlp_train = DataLoader(charger_tenseurs("train"), batch_size=64, shuffle=True)
mlp = nn.Sequential(nn.Linear(3072, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
for epoch in range(20):
    train_one_epoch(mlp, mlp_train, nn.CrossEntropyLoss(), optimizer)

In [ ]:
mlp_val = DataLoader(charger_tenseurs("val"), batch_size=256)
print(f"MLP (pixels aplatis) : accuracy val {evaluate(mlp, mlp_val, loss_fn)[1]:.3f}")
print(f"CNN                  : accuracy val {evaluate(model, val_loader, loss_fn)[1]:.3f}")

Question : d'où vient l'écart ? Qu'est-ce que la convolution exploite que le MLP ne voit pas ?

## Bonus

**B1.** Calculez la matrice de confusion du CNN sur la validation : ligne = vraie classe, colonne = classe prédite. Quelles classes sont le plus souvent confondues ?

In [ ]:
confusion = torch.zeros(10, 10, dtype=torch.int64)
model.eval()
with torch.no_grad():
    for x, y in val_loader:
        for vraie, predite in zip(y, model(x.to(device)).argmax(1).cpu()):
            # TODO : ajouter 1 dans la case [vraie, predite]
            ...
print(val_ds.classes)
print(confusion)

**B2.** Réentraînez le même CNN sans augmentation. L'écart avec l'entraînement change-t-il ?

In [ ]:
# TODO : même entraînement sans augmentation